**IMPORTANT: CLEAR OUTPUTS BEFORE PUSHING**

## Pre-Processing

In [ ]:
import pandas as pd
import numpy as np

PARTICIPANTS_FILEPATH = "input/participants.csv"
POST_RATINGS_FILEPATH = "input/post_ratings.csv"
POST_ATTRIBUTES_FILEPATH = "input/post_attributes.csv"

participants_df = pd.read_csv(PARTICIPANTS_FILEPATH)
post_ratings_df = pd.read_csv(POST_RATINGS_FILEPATH)
post_attributes_df = pd.read_csv(POST_ATTRIBUTES_FILEPATH)

In [ ]:
post_ratings_df_valid = post_ratings_df[(post_ratings_df['blank_frame'] == False)]
print(len(post_ratings_df_valid))

## Preliminary Analysis and Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('default')
style = {
    'boxprops': {'color': 'tab:blue'},
    'whiskerprops': {'color': 'tab:blue'},
    'medianprops': {'color': 'tab:green'}
}
enjoyment_boxplot = post_ratings_df_valid.boxplot(column='rate_enjoy_n', by='condition_served', **style)
plt.savefig("post_enjoyment_boxplot.png", bbox_inches='tight', dpi=300)
informativeness_boxplot = post_ratings_df_valid.boxplot(column='rate_inform_n', by='condition_served', **style)
plt.savefig("post_informativeness_boxplot.png", bbox_inches='tight', dpi=300)
satisfaction_boxplot = post_ratings_df_valid.boxplot(column='rate_satisfied_n', by='condition_served', **style)
plt.savefig("post_satisfaction_boxplot.png", bbox_inches='tight', dpi=300)

In [ ]:
post_enjoy_crosstab = pd.crosstab(
    index=post_ratings_df_valid['condition_served'],
    columns=post_ratings_df_valid['rate_enjoy_n'],
)
post_enjoy_crosstab_pct = post_enjoy_crosstab.div(post_enjoy_crosstab.sum(axis=1), axis=0) * 100
print(post_enjoy_crosstab_pct.head(10))
post_enjoy_crosstab_pct.plot(kind='bar')

In [ ]:
plt.style.use('default')
style = {
    'boxprops': {'color': 'tab:blue'},
    'whiskerprops': {'color': 'tab:blue'},
    'medianprops': {'color': 'tab:green'}
}
enjoyment_boxplot = participants_df.boxplot(column='post_Q4.1_n', by='condition', **style)
plt.savefig("participant_enjoyment_boxplot.png", bbox_inches='tight', dpi=300)
informativeness_boxplot = participants_df.boxplot(column='post_Q4.3_n', by='condition', **style)
plt.savefig("participant_informativeness_boxplot.png", bbox_inches='tight', dpi=300)
satisfaction_boxplot = participants_df.boxplot(column='post_Q4.5_n', by='condition', **style)
plt.savefig("participant_satisfaction_boxplot.png", bbox_inches='tight', dpi=300)

In [ ]:
participant_enjoy_crosstab = pd.crosstab(
    index=participants_df['condition'],
    columns=participants_df['post_Q4.11_n'],
)
participant_enjoy_crosstab_pct = participant_enjoy_crosstab.div(participant_enjoy_crosstab.sum(axis=1), axis=0) * 100
print(participant_enjoy_crosstab_pct.head(10))
participant_enjoy_crosstab_pct.plot(kind='bar')

In [ ]:
participants_df.boxplot(column='post_Q4.11_n', by='condition', **style)

## Statistical Significance

In [ ]:
import statsmodels.formula.api as smf

def record_pvals(results, reference_group, independent_var, dependent_var, pvals):
    if dependent_var not in pvals:
        pvals[dependent_var] = {}
    for treatment_group in ['control', 'agreeable', 'neutral', 'satirical']:
        if treatment_group != reference_group:
            # Generate standard ordering for each group pair to avoid adding duplicates
            comparison_groups = [reference_group, treatment_group]
            comparison_groups.sort()
            pval_key = f'{comparison_groups[0]}-{comparison_groups[1]}'

            # Add pval to dict
            if pval_key not in pvals[dependent_var]:
                results_key = f"C({independent_var}, Treatment(reference='{reference_group}'))[T.{treatment_group}]"
                pvals[dependent_var][pval_key] = results.pvalues[results_key]

def model_mixedlm(data, reference_group, independent_var, dependent_var):
    model_intercept = smf.mixedlm(
        f"{dependent_var} ~ C({independent_var}, Treatment(reference='{reference_group}'))",
        data,
        groups=data['pid']
    )
    result_intercept = model_intercept.fit()

    print()
    print(f"Reference Group: {reference_group} | Dependent Variable: {dependent_var}")
    print(result_intercept.summary())

    return result_intercept

def model_ols(data, reference_group, independent_var, dependent_var):
    model_intercept = smf.ols(
        f"{dependent_var} ~ C({independent_var}, Treatment(reference='{reference_group}'))",
        data
    )
    result_intercept = model_intercept.fit()

    print()
    print(f"Reference Group: {reference_group} | Dependent Variable: {dependent_var}")
    print(result_intercept.summary())

    return result_intercept

### Post-Level

In [ ]:
post_pvals = {}
for reference_group in ['control', 'agreeable', 'neutral', 'satirical']:
    for dependent_var in ['rate_enjoy_n', 'rate_inform_n', 'rate_satisfied_n']:
        results = model_mixedlm(
            data=post_ratings_df_valid,
            reference_group=reference_group,
            independent_var='condition_served',
            dependent_var=dependent_var,
        )
        record_pvals(
            results=results,
            reference_group=reference_group,
            independent_var='condition_served',
            dependent_var=dependent_var,
            pvals=post_pvals
        )
print(post_pvals)

### Participant-Level

In [ ]:
participants_df = participants_df.rename(columns={
    'post_Q4.1_n': 'enjoy_n',
    'post_Q4.3_n': 'inform_n',
    'post_Q4.5_n': 'satisfied_n',
    'post_Q4.11_n': 'engage_offline_n'
})

participant_pvals = {}
for reference_group in ['control', 'agreeable', 'neutral', 'satirical']:
    for dependent_var in ['enjoy_n', 'inform_n', 'satisfied_n', 'engage_offline_n']:
        results = model_ols(
            data=participants_df,
            reference_group=reference_group,
            independent_var='condition',
            dependent_var=dependent_var,
        )
        record_pvals(
            results=results,
            reference_group=reference_group,
            independent_var='condition',
            dependent_var=dependent_var,
            pvals=participant_pvals
        )
print(participant_pvals)

### Post-Level with Political Alignment

In [ ]:
participants_df['participant_polarity'] = np.where(
    participants_df['party'] == 'D',
    participants_df['pre_Q1.2_n'] / -5,
    participants_df['pre_Q1.2_n'] / 5
)

full_df = pd.merge(post_ratings_df, participants_df, on='pid', how='left')
full_df['alignment'] = full_df['polarity_score'] * full_df['participant_polarity']

In [ ]:
import statsmodels.formula.api as smf

def model_alignment_mixedlm(data, reference_group, independent_var, dependent_var):
    model_intercept = smf.mixedlm(
        f"{dependent_var} ~ alignment * C({independent_var}, Treatment(reference='{reference_group}'))",
        data,
        groups=data['pid']
    )
    result_intercept = model_intercept.fit()

    print()
    print(f"Reference Group: {reference_group} | Dependent Variable: {dependent_var}")
    print(result_intercept.summary())

    return result_intercept

post_alignment_pvals = {}
for reference_group in ['control', 'agreeable', 'neutral', 'satirical']:
    for dependent_var in ['rate_enjoy_n', 'rate_inform_n', 'rate_satisfied_n']:
        results = model_mixedlm(
            data=full_df.dropna(),
            reference_group=reference_group,
            independent_var='condition_served',
            dependent_var=dependent_var,
        )
        record_pvals(
            results=results,
            reference_group=reference_group,
            independent_var='condition_served',
            dependent_var=dependent_var,
            pvals=post_alignment_pvals
        )
print(post_alignment_pvals)